# Prime Numbers Lab — Notebook 12: Normalized Gap Distribution

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** Measure normalized prime gaps against the exponential baseline and quantify distributional fit across scale windows.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

## 0. Setup

This notebook follows the shared Prime Numbers Lab template:

1. define one constraint  
2. generate or load one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and optional TeX  
6. package results into a root-level export zip

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import sympy as sp
    HAS_SYMPY = True
except ImportError:
    HAS_SYMPY = False

NOTEBOOK_ID = "12_normalized_gap_distribution"
NOTEBOOK_TITLE = "Normalized Gap Distribution"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")
print(f"SymPy available: {HAS_SYMPY}")

## 1. Premise

Prime gaps have a classical scale baseline: near size `x`, typical gaps are on the order of `log(x)`.

This notebook measures the normalized gap variable

\[
z = \frac{g}{\log(x)}
\]

where \(g = p_{n+1}-p_n\), and compares empirical normalized gaps to an exponential baseline.

**Short vocabulary:**

- **Remains under constraint / persists:** normalized gap statistics stabilize across larger windows.
- **Drift:** measured departure from an exponential or log-scale baseline.
- **Recoverability:** ability to recover stable distributional shape after scale normalization.

## 2. Constraint definition

For consecutive primes \(p_n, p_{n+1}\), define:

\[
g_n = p_{n+1} - p_n
\]

Use local scale proxy:

\[
x_n = p_n
\]

Normalize gaps by expected local size:

\[
z_n = \frac{g_n}{\log(x_n)}
\]

The baseline comparison is the exponential distribution:

\[
P(Z > t) = e^{-t}, \qquad Z \sim \mathrm{Exp}(1)
\]

The notebook asks whether normalized gaps become more distributionally stable as scale increases.

In [ ]:
N_MAX = 1_000_000
RANDOM_SEED = 9423
MIN_X_FOR_GAPS = 100
N_WINDOWS = 16
HIST_BINS = np.linspace(0, 8, 41)
TAIL_THRESHOLDS = [1, 2, 3, 4, 5]

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "MIN_X_FOR_GAPS": MIN_X_FOR_GAPS,
    "N_WINDOWS": N_WINDOWS,
    "HIST_BINS": HIST_BINS.tolist(),
    "TAIL_THRESHOLDS": TAIL_THRESHOLDS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Data generation or loading

Generate primes directly up to `N_MAX`, then compute consecutive prime gaps and normalized gaps.

In [ ]:
def generate_primes(n_max: int) -> np.ndarray:
    """Generate primes less than n_max."""
    if n_max < 2:
        return np.array([], dtype=int)

    if HAS_SYMPY:
        return np.array(list(sp.primerange(2, n_max)), dtype=int)

    sieve = np.ones(n_max, dtype=bool)
    sieve[:2] = False
    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max:i] = False
    return np.nonzero(sieve)[0].astype(int)


primes = generate_primes(N_MAX)

p_left = primes[:-1]
p_right = primes[1:]
gaps = p_right - p_left

mask = p_left >= MIN_X_FOR_GAPS
x = p_left[mask].astype(float)
gap = gaps[mask].astype(float)
logx = np.log(x)
z = gap / logx

gap_df = pd.DataFrame({
    "p": x.astype(int),
    "next_p": p_right[mask].astype(int),
    "gap": gap.astype(int),
    "log_x": logx,
    "z_gap_over_logx": z,
})

summary = {
    "n_max": N_MAX,
    "prime_count": int(len(primes)),
    "gap_count": int(len(gap_df)),
    "min_x_for_gaps": MIN_X_FOR_GAPS,
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist() if len(primes) >= 10 else primes.tolist(),
}

gap_df.head(), summary

## 4. Measurement

Measurements:

- normalized gap histogram versus \(\mathrm{Exp}(1)\)
- empirical CDF versus exponential CDF
- windowed mean and spread relative to \(\log(x)\)
- distributional distance to exponential baseline
- scale-window heatmap of normalized gap distributions
- quantiles and tail exceedance rates

In [ ]:
def make_log_windows(values: np.ndarray, n_windows: int) -> np.ndarray:
    """Return log-spaced window edges covering positive values."""
    lo = max(float(values.min()), 2.0)
    hi = float(values.max()) + 1.0
    return np.geomspace(lo, hi, n_windows + 1)


def exp_pdf(t: np.ndarray) -> np.ndarray:
    return np.exp(-t)


def exp_cdf(t: np.ndarray) -> np.ndarray:
    return 1.0 - np.exp(-t)


def js_divergence_from_hist(hist_density: np.ndarray, bin_edges: np.ndarray) -> float:
    """Approximate JS divergence between empirical histogram and Exp(1)."""
    widths = np.diff(bin_edges)
    p = hist_density * widths
    p = p / max(p.sum(), 1e-12)

    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    q_density = exp_pdf(centers)
    q = q_density * widths
    q = q / max(q.sum(), 1e-12)

    eps = 1e-12
    p = np.clip(p, eps, None)
    q = np.clip(q, eps, None)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


window_edges = make_log_windows(x, N_WINDOWS)

window_rows = []
hist_rows = []
tail_rows = []

for i in range(len(window_edges) - 1):
    lo, hi = window_edges[i], window_edges[i + 1]
    m = (x >= lo) & (x < hi)
    if m.sum() < 5:
        continue

    z_w = z[m]
    gap_w = gap[m]
    logx_w = logx[m]
    midpoint = math.sqrt(lo * hi)

    hist_density, bin_edges = np.histogram(z_w, bins=HIST_BINS, density=True)
    js = js_divergence_from_hist(hist_density, bin_edges)
    distributional_score = 1.0 / (1.0 + js)

    row = {
        "window_index": i,
        "x_low": float(lo),
        "x_high": float(hi),
        "window_midpoint": float(midpoint),
        "n_gaps": int(m.sum()),
        "mean_gap": float(np.mean(gap_w)),
        "std_gap": float(np.std(gap_w)),
        "mean_logx": float(np.mean(logx_w)),
        "mean_gap_over_mean_logx": float(np.mean(gap_w) / np.mean(logx_w)),
        "std_gap_over_mean_logx": float(np.std(gap_w) / np.mean(logx_w)),
        "mean_z": float(np.mean(z_w)),
        "std_z": float(np.std(z_w)),
        "q50_z": float(np.quantile(z_w, 0.50)),
        "q75_z": float(np.quantile(z_w, 0.75)),
        "q90_z": float(np.quantile(z_w, 0.90)),
        "q95_z": float(np.quantile(z_w, 0.95)),
        "q99_z": float(np.quantile(z_w, 0.99)),
        "js_divergence_to_exp1": js,
        "distributional_score": float(distributional_score),
    }
    window_rows.append(row)

    for b, val in enumerate(hist_density):
        hist_rows.append({
            "window_index": i,
            "window_midpoint": float(midpoint),
            "bin_left": float(bin_edges[b]),
            "bin_right": float(bin_edges[b + 1]),
            "bin_center": float(0.5 * (bin_edges[b] + bin_edges[b + 1])),
            "density": float(val),
        })

    for t in TAIL_THRESHOLDS:
        tail_rows.append({
            "window_index": i,
            "window_midpoint": float(midpoint),
            "threshold": float(t),
            "empirical_tail_probability": float(np.mean(z_w > t)),
            "exp1_tail_probability": float(math.exp(-t)),
        })


window_df = pd.DataFrame(window_rows)
hist_df = pd.DataFrame(hist_rows)
tail_df = pd.DataFrame(tail_rows)

measurement = {
    "overall_mean_z": float(np.mean(z)),
    "overall_std_z": float(np.std(z)),
    "overall_q50_z": float(np.quantile(z, 0.50)),
    "overall_q90_z": float(np.quantile(z, 0.90)),
    "overall_q99_z": float(np.quantile(z, 0.99)),
    "final_window_distributional_score": float(window_df["distributional_score"].iloc[-1]),
    "mean_distributional_score": float(window_df["distributional_score"].mean()),
    "max_distributional_score": float(window_df["distributional_score"].max()),
    "min_js_divergence_to_exp1": float(window_df["js_divergence_to_exp1"].min()),
    "final_mean_gap_over_mean_logx": float(window_df["mean_gap_over_mean_logx"].iloc[-1]),
}

window_df.tail(), measurement

## 5. CGCS score

Working notebook-specific definition:

\[
CGCS = \frac{1}{1 + JS(P_{empirical}(z), P_{Exp(1)}(z))}
\]

where \(JS\) is Jensen–Shannon divergence computed from normalized-gap histogram probabilities.

This makes the score high when empirical normalized gaps are close to the exponential baseline.

In [ ]:
cgcs_score = measurement["final_window_distributional_score"]

cgcs = {
    "score": float(cgcs_score),
    "definition": "1 / (1 + Jensen-Shannon divergence between final-window normalized gap distribution and Exp(1) baseline)",
    "interpretation": "Closer to 1 indicates stronger final-window distributional agreement after z = gap/log(x) normalization.",
}

cgcs

## 6. Visualization

Figures are saved to the notebook-local export path:

`12_normalized_gap_distribution/figures/`

In [ ]:
figure_paths = []

# Figure 1: normalized gap histogram versus exponential pdf
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(z, bins=HIST_BINS, density=True, alpha=0.5, label="empirical normalized gaps")
grid = np.linspace(0, 8, 400)
ax.plot(grid, exp_pdf(grid), linewidth=2.5, label="Exponential(1) pdf")
ax.set_title("Normalized prime gaps versus exponential baseline")
ax.set_xlabel("normalized gap z = gap / log(x)")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_normalized_gap_histogram.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 2: ECDF
z_sorted = np.sort(z)
ecdf_y = np.arange(1, len(z_sorted) + 1) / len(z_sorted)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z_sorted, ecdf_y, label="empirical ECDF")
ax.plot(grid, exp_cdf(grid), linestyle="--", linewidth=2.5, label="Exponential(1) CDF")
ax.set_title("ECDF of normalized gaps")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("cumulative probability")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_normalized_gap_ecdf.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 3: mean and spread scaling
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(window_df["window_midpoint"], window_df["mean_gap_over_mean_logx"], marker="o", label="mean(gap) / mean(log x)")
ax.plot(window_df["window_midpoint"], window_df["std_gap_over_mean_logx"], marker="o", label="std(gap) / mean(log x)")
ax.axhline(1.0, linestyle="--", label="Exp(1) reference")
ax.set_xscale("log")
ax.set_title("Mean and spread scaling relative to log(x)")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("normalized value")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_spread_scaling.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 4: heatmap by x-window
pivot = hist_df.pivot_table(index="window_index", columns="bin_center", values="density", aggfunc="mean")
y_labels = []
for _, row in window_df.iterrows():
    y_labels.append(f"{int(row['x_low'])}-{int(row['x_high'])}")

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(pivot.values, aspect="auto", origin="lower")
ax.set_title("Normalized gap distribution heatmap by x-window")
ax.set_xlabel("normalized gap z = gap / log(x)")
ax.set_ylabel("x window")

x_tick_idx = np.linspace(0, len(pivot.columns) - 1, 9).astype(int)
ax.set_xticks(x_tick_idx)
ax.set_xticklabels([f"{pivot.columns[i]:.1f}" for i in x_tick_idx])

y_tick_idx = np.arange(len(y_labels))
ax.set_yticks(y_tick_idx)
ax.set_yticklabels(y_labels)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("density")
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_normalized_gap_heatmap.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 5: distributional score
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(window_df["window_midpoint"], window_df["distributional_score"], marker="o", label="distributional score")
ax.set_xscale("log")
ax.set_title("Distributional fit score by scale window")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("score")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_distributional_fit_score.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 6: quantiles
exp_quantiles = {
    "q50_z": -math.log(1 - 0.50),
    "q75_z": -math.log(1 - 0.75),
    "q90_z": -math.log(1 - 0.90),
    "q95_z": -math.log(1 - 0.95),
    "q99_z": -math.log(1 - 0.99),
}

fig, ax = plt.subplots(figsize=(10, 6))
for qcol in ["q50_z", "q75_z", "q90_z", "q95_z", "q99_z"]:
    ax.plot(window_df["window_midpoint"], window_df[qcol], marker="o", label=qcol)
    ax.axhline(exp_quantiles[qcol], linestyle="--", alpha=0.5)

ax.set_xscale("log")
ax.set_title("Normalized gap quantiles by scale window")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("quantile of gap / log(x)")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_normalized_gap_quantiles.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 7: tail exceedance rates
fig, ax = plt.subplots(figsize=(10, 6))
for t in TAIL_THRESHOLDS:
    sub = tail_df[tail_df["threshold"] == t]
    ax.plot(sub["window_midpoint"], sub["empirical_tail_probability"], marker="o", label=f"emp P(z>{t})")
    ax.axhline(math.exp(-t), linestyle="--", alpha=0.5)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Tail exceedance rates for normalized gaps")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("tail probability")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_tail_exceedance_rates.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

# Figure 8: JS divergence
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(window_df["window_midpoint"], window_df["js_divergence_to_exp1"], marker="o", label="JS divergence to Exp(1)")
ax.set_xscale("log")
ax.set_title("Windowed distribution distance to exponential baseline")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("Jensen-Shannon divergence")
ax.legend()
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / f"{NOTEBOOK_NUM}_js_divergence_to_exp1.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
figure_paths.append(fig_path)
plt.show()

figure_paths

## 7. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**

Avoid claiming this notebook proves RH or replaces analytic number theory. The goal is measurement clarity.

In [ ]:
interpretation = f"""
# {NOTEBOOK_TITLE} (Notebook {NOTEBOOK_NUM})

## Constraint result

This notebook tests whether prime gaps become more stable after local scale normalization:

$$
z_n = \\frac{{p_{{n+1}} - p_n}}{{\\log(p_n)}}.
$$

The comparison baseline is:

$$
Z \\sim \\mathrm{{Exp}}(1).
$$

## Remains under constraint

The mean normalized gap remains close to 1 at larger scale windows:

$$
\\frac{{\\mathbb{{E}}[g]}}{{\\mathbb{{E}}[\\log x]}} \\approx 1.
$$

This supports the expected log-scale normalization:

$$
g \\sim \\log x.
$$

## Drift

Distributional drift remains visible in quantiles, tails, and histogram shape. The distributional score is based on Jensen--Shannon divergence:

$$
CGCS = \\frac{{1}}{{1 + JS(P_{{empirical}}, P_{{Exp(1)}})}}.
$$

Final-window CGCS score:

$$
CGCS_{{final}} = {cgcs_score:.6f}.
$$

Mean distributional score:

$$
\\overline{{CGCS}} = {measurement["mean_distributional_score"]:.6f}.
$$

## Recoverability

The normalized-gap distribution becomes more comparable across scale windows after dividing by $\\log(x)$. This indicates that gap shape is partially recoverable from a local scale correction, even when raw gaps grow with $x$.

## Caution

This notebook does not prove a theorem about prime gaps or the Riemann Hypothesis. It provides a reproducible empirical diagnostic for log-normalized gap structure and its distributional drift.

## Core result

The measurable result is:

$$
z = \\frac{{g}}{{\\log(x)}} \\quad \\text{{has a scale-stabilized distribution with an exponential-style baseline.}}
$$

## Summary

- gap size grows with local $\\log(x)$ scale
- normalized mean stays near 1
- distributional fit generally improves across larger scale windows
- tails and quantiles preserve structured drift
- CGCS measures closeness to the exponential baseline

Constraint → signal > noise
""".strip()

print(interpretation)

## 8. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summary
- window metrics CSV
- normalized gap sample CSV
- histogram CSV
- tail exceedance CSV
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

csv_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
gaps_csv_path = DATA_DIR / f"{NOTEBOOK_NUM}_normalized_gaps_sample.csv"
windows_csv_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_metrics.csv"
hist_csv_path = DATA_DIR / f"{NOTEBOOK_NUM}_histogram_by_window.csv"
tail_csv_path = DATA_DIR / f"{NOTEBOOK_NUM}_tail_exceedance.csv"
json_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(csv_path, index=False)
gap_df.sample(min(5000, len(gap_df)), random_state=RANDOM_SEED).sort_values("p").to_csv(gaps_csv_path, index=False)
window_df.to_csv(windows_csv_path, index=False)
hist_df.to_csv(hist_csv_path, index=False)
tail_df.to_csv(tail_csv_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(csv_path),
        "normalized_gaps_sample": str(gaps_csv_path),
        "window_metrics": str(windows_csv_path),
        "histogram_by_window": str(hist_csv_path),
        "tail_exceedance": str(tail_csv_path),
    },
    "docs": {
        "interpretation": str(interpretation_md_path),
        "design_notes": str(design_notes_md_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

json_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

figures_md = "\n\n## Figures\n\n"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_', ' ').title()}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")

design_notes = f"""
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook {NOTEBOOK_NUM} moves from reconstruction and window-stability diagnostics into normalized prime-gap distribution diagnostics.

## Constraint

Normalize consecutive prime gaps by local log scale:

$$
z_n = \\frac{{p_{{n+1}} - p_n}}{{\\log(p_n)}}.
$$

## Measurement

Measure histogram, ECDF, quantiles, tail exceedance, and Jensen--Shannon distance to an Exp(1) baseline.

## CGCS score

Notebook-specific CGCS:

$$
CGCS = \\frac{{1}}{{1 + JS(P_{{empirical}}, P_{{Exp(1)}})}}.
$$

The final-window score is `{cgcs_score:.6f}`.

## Handoff

Next notebook should connect normalized gap distribution diagnostics back to sieve/reconstruction diagnostics:

- compare reconstructed candidates versus true primes under normalized-gap statistics
- test whether local reconstruction preserves Exp(1)-style normalized gaps
- quantify which reconstruction method best preserves both density and gap distribution
""".strip()

design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf"""
\section*{{{NOTEBOOK_TITLE}}}

Notebook {NOTEBOOK_NUM} measures normalized prime gaps against an exponential baseline.

\begin{{itemize}}
  \item Prime count below $N$: {len(primes)}
  \item Gap count after cutoff: {len(gap_df)}
  \item Overall mean normalized gap: {measurement["overall_mean_z"]:.6f}
  \item Overall standard deviation of normalized gap: {measurement["overall_std_z"]:.6f}
  \item Final-window CGCS score: {cgcs_score:.6f}
\end{{itemize}}

The normalized gap variable is

\[
z_n = \frac{{p_{{n+1}} - p_n}}{{\log(p_n)}}.
\]

The baseline comparison is

\[
Z \sim \mathrm{{Exp}}(1).
\]
""".strip()

summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf"""
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: {NOTEBOOK_TITLE}}}

\subsection*{{Prime gaps}}

For consecutive primes $p_n$ and $p_{{n+1}}$,

\[
g_n = p_{{n+1}} - p_n.
\]

\subsection*{{Local log-scale normalization}}

At local scale $x_n = p_n$,

\[
z_n = \frac{{g_n}}{{\log(x_n)}}.
\]

\subsection*{{Exponential baseline}}

A standard random-model comparison is

\[
Z \sim \mathrm{{Exp}}(1),
\]

with density and cumulative distribution

\[
f(z) = e^{{-z}}, \qquad F(z) = 1 - e^{{-z}}.
\]

Tail exceedance is

\[
P(Z > t) = e^{{-t}}.
\]

\subsection*{{Distributional CGCS}}

For empirical normalized-gap distribution $P_{{empirical}}$ and exponential baseline $P_{{Exp(1)}}$,

\[
CGCS = \frac{{1}}{{1 + JS(P_{{empirical}}, P_{{Exp(1)}})}}.
\]

Notebook {NOTEBOOK_NUM} final-window score:

\[
CGCS_{{final}} = {cgcs_score:.6f}.
\]

\subsection*{{Interpretation}}

Structure remains under constraint when raw gap growth is reduced to a comparatively stable normalized distribution. Drift remains visible in tails and quantiles, so the result is a diagnostic rather than a theorem.

\end{{document}}
""".strip()

math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

csv_path, json_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path

## 9. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 10. Next notebook handoff

Next: connect normalized gap distribution to reconstruction quality.

Notebook 13 can test whether candidate reconstruction methods preserve:

- local density
- normalized gap distribution
- tail exceedance structure
- windowed CGCS under partial observation or corruption

In [ ]:
next_step = (
    "Next: test whether reconstructed candidate sets preserve normalized gap "
    "statistics, tail structure, and distributional CGCS across scale windows."
)
print(next_step)